# QM640 · 08 Instrument I4: Jev fast decision model (RQ1, with bulk run for RQ2/RQ4)
Needs the outputs of notebook 04. CPU runtime; no GPU units.

**Before running:** in Colab click the key icon (Secrets) on the left, add a secret named `JEV_API_KEY` with your TypeSafe API key, and switch on notebook access. The key is never written into the notebook.

*Runtime → Run all.* Part A takes minutes; Part B runs until the budget is used.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
JEV_KEY = userdata.get('JEV_API_KEY')
MODEL = 'jev-1.13.0'     # pinned version for reproducibility
BUDGET_USD = 4.80        # stop before the free credit is used up
MAX_PER_MONTH = 200      # bulk run: issues per repository-month

## 1. Settings
Each ticket is sent once with four questions (D1 type, D1 priority, D4 resolution, D5 component). Jev returns a choice, a probability for every option and a confidence. The allowed answers are the label spaces saved by notebook 04.

In [ ]:
import json, time, threading, datetime, random, numpy as np, pandas as pd, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
JIRA = DATA_ROOT/'processed'/'jira'; TESTS = DATA_ROOT/'processed'/'jira_tests'
RESULTS = DATA_ROOT/'results'; RAW = RESULTS/'jev_raw'; RAW.mkdir(parents=True, exist_ok=True)
API = 'https://api.typesafe.ai/v1/systemone'
PRICE_PER_M_INPUT = 0.042            # US$ per million input tokens; output tokens are free
SEED = 640
LABELS = json.load(open(TESTS/'label_spaces.json'))
QUESTIONS = {   # one call answers all four classification decisions for a ticket
  'D1_issuetype': ('What type of issue is this?', LABELS['D1_issuetype']),
  'D1_priority': ('What priority should this issue be given?', LABELS['D1_priority']),
  'D4_resolution': ('How is this issue most likely to be resolved?', LABELS['D4_resolution']),
  'D5_component': ('Which project component (team) should handle this issue?', LABELS['D5_component'])}
def txt(x): return x if isinstance(x, str) else ''
def ticket_state(r, n_desc=1200):
    return f"Project: {r['project']}\nSummary: {txt(r['summary'])}\nDescription: {txt(r['description'])[:n_desc]}"
def cls_questions():
    return {q: {'type': 'choice', 'instructions': ins, 'criteria': {lab: None for lab in labs}} for q, (ins, labs) in QUESTIONS.items()}

## 2. API client
Retries with back-off on rate limits (HTTP 429) and server errors. Stops at the budget or if the credit runs out (HTTP 402). Every answer is saved to `results/jev_raw/*.jsonl`. If the session stops, run all cells again; finished tickets are skipped.

In [ ]:
class Jev:
    def __init__(self, key, budget_usd):
        self.s = requests.Session(); self.s.headers.update({'Authorization': f'Bearer {key}', 'Content-Type': 'application/json'})
        self.budget, self.tokens, self.calls, self.lock, self.stop = budget_usd, 0, 0, threading.Lock(), None
    @property
    def spent(self): return self.tokens*PRICE_PER_M_INPUT/1e6
    def ask(self, state, questions):
        if self.stop: return None
        body = {'model': MODEL, 'state': state, 'questions': questions}
        for attempt in range(8):
            try:
                r = self.s.post(API, json=body, timeout=60)
            except requests.RequestException:
                time.sleep(min(60, 2**attempt)); continue
            if r.status_code == 200:
                out = r.json(); t = out.get('usage', {}).get('input_tokens', 0)
                with self.lock:
                    self.tokens += t; self.calls += 1
                    if self.spent >= self.budget: self.stop = f'budget of ${self.budget} reached'
                return out
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(min(60, 2**attempt) + random.random()); continue
            if r.status_code in (401, 402, 403):
                self.stop = f'HTTP {r.status_code}: {r.text[:200]}'; return None
            return {'error': r.status_code, 'detail': r.text[:500]}     # 422 etc.: logged and skipped
        return {'error': 'retries exhausted'}
def run_jobs(jev, jobs, log_file, workers=8):
    """jobs: list of (job_id, state, questions). Results appended to a JSON-lines file; finished jobs are skipped on rerun."""
    done = set()
    if log_file.exists():
        for line in open(log_file):
            o = json.loads(line)
            if 'answers' in o: done.add(o['job_id'])
    todo = [j for j in jobs if j[0] not in done]
    print(f'{log_file.name}: {len(done)} already done, {len(todo)} to run')
    lock, buf, t0 = threading.Lock(), [], time.time()
    def flush():
        with open(log_file, 'a') as f:
            for o in buf: f.write(json.dumps(o) + '\n')
        buf.clear()
    with ThreadPoolExecutor(workers) as ex:
        futs = {ex.submit(jev.ask, s, q): jid for jid, s, q in todo}
        for n, fu in enumerate(as_completed(futs), 1):
            out = fu.result()
            if out is not None:
                out['job_id'] = futs[fu]; out['ts'] = time.time()
                with lock:
                    buf.append(out)
                    if len(buf) >= 100: flush()
            if n % 500 == 0: print(f'  {n}/{len(todo)} | {jev.calls} calls | ${jev.spent:.3f} | {n/(time.time()-t0):.1f}/s')
            if jev.stop:
                for f in futs: f.cancel()
                break
    with lock: flush()
    print(f'finished: {jev.calls} calls this session, ${jev.spent:.3f} spent', '| STOPPED: ' + jev.stop if jev.stop else '')
def read_log(log_file):
    rows = []
    for line in open(log_file):
        o = json.loads(line)
        if 'answers' in o: rows.append(o)
    return rows

## 3. Connect and test with one ticket
Shows the model list, one full answer and the measured tokens per ticket.

In [ ]:
jev = Jev(JEV_KEY, BUDGET_USD)
jev.tokens = sum(json.loads(l).get('usage', {}).get('input_tokens', 0) for f in RAW.glob('*.jsonl') for l in open(f))   # spend from earlier sessions counts towards the budget
print(f'already spent in earlier sessions: ${jev.spent:.3f}')
try:
    models = requests.get('https://api.typesafe.ai/v1/models', headers={'Authorization': f'Bearer {JEV_KEY}'}, timeout=30).json()
    print('models:', models)
except Exception as e: print('model list not available:', e)
tick = pd.read_parquet(TESTS/'jira_test_tickets.parquet').drop_duplicates('key').set_index('key', drop=False)
probe = jev.ask(ticket_state(tick.iloc[0]), cls_questions())
if 'usage' not in (probe or {}):   # pinned version not available: fall back to the rolling alias
    print('pinned model rejected:', probe); MODEL = 'jev-latest'
    probe = jev.ask(ticket_state(tick.iloc[0]), cls_questions())
print(json.dumps(probe, indent=1)[:1500])
TOK_PER_CALL = probe['usage']['input_tokens']
print(f'{TOK_PER_CALL} input tokens per ticket (four decisions) = ${TOK_PER_CALL*PRICE_PER_M_INPUT/1e6:.6f} per ticket')

## 4. Part A — RQ1 test sets
A1: the four 385-ticket classification test sets (about 1,500 unique tickets).

In [ ]:
cls_keys = sorted(set().union(*[set(pd.read_csv(TESTS/f'testset_jira_{t}.csv')['key']) for t in QUESTIONS]))
jobs = [(k, ticket_state(tick.loc[k]), cls_questions()) for k in cls_keys]
run_jobs(jev, jobs, RAW/'rq1_classification.jsonl')

A2: 200 tickets asked two more times, to measure run-to-run consistency (repeatability of the instrument).

In [ ]:
rep_keys = sorted(random.Random(SEED).sample(cls_keys, 200))
for rep in (2, 3):
    run_jobs(jev, [(f'{k}#rep{rep}', ticket_state(tick.loc[k]), cls_questions()) for k in rep_keys], RAW/f'rq1_repeat{rep}.jsonl')

A3: duplicate and cause search. Jev picks the matching issue from the I1 top-10 shortlist (re-ranking), so the result is bounded by I1 Recall@10.

In [ ]:
need = set()
for t in ('D2_duplicate', 'D6_cause'):
    need |= {k for ks in pd.read_csv(TESTS/f'i1_predictions_{t}.csv')['i1_top10'] for k in json.loads(ks)}
need -= set(tick.index)
extra = pd.concat([pd.read_parquet(f, columns=['key', 'project', 'summary', 'description'], filters=[('key', 'in', list(need))]) for f in sorted(JIRA.glob('*.parquet'))])
texts = pd.concat([tick[['key', 'project', 'summary', 'description']], extra]).drop_duplicates('key').set_index('key')
INSTR = {'D2_duplicate': 'Which of the earlier issues listed does this new issue duplicate?',
         'D6_cause': 'Which of the earlier issues listed caused the problem reported in this new issue?'}
def short(k): r = texts.loc[k]; return f"{txt(r['summary'])} | {txt(r['description'])[:200]}"
for t in ('D2_duplicate', 'D6_cause'):
    pr = pd.read_csv(TESTS/f'i1_predictions_{t}.csv'); jobs = []
    for k, ks in zip(pr['key'], pr['i1_top10']):
        cands = json.loads(ks)
        crit = {f'c{i+1}': short(c) for i, c in enumerate(cands)}; crit['none'] = 'None of the listed issues'
        jobs.append((k, 'New issue:\n' + ticket_state(texts.loc[k]), {t: {'type': 'choice', 'instructions': INSTR[t], 'criteria': crit}}))
    run_jobs(jev, jobs, RAW/f'rq1_{t}.jsonl')
print(f'Part A spent ${jev.spent:.3f}')

## 5. Part B — Bulk run over the whole timeline
Uses the remaining credit on up to `MAX_PER_MONTH` issues per repository-month, excluding test tickets. The answers feed RQ2 threshold tuning and the RQ4 month-by-month replay. Issues are taken in rounds across all months, so if the budget runs out every month is still covered evenly. This is the long step (hours); it can be stopped and resumed.

In [ ]:
# Issues are sampled round-robin across repository-months, so if the budget runs out every month is still covered evenly.
excluded = set(tick.index)
bulk = pd.concat([pd.read_parquet(f, columns=['repository', 'key', 'project', 'created', 'summary', 'description']) for f in sorted(JIRA.glob('*.parquet'))])
bulk['created'] = pd.to_datetime(bulk['created'], errors='coerce', utc=True)
bulk = bulk[(bulk['created'] >= pd.Timestamp('2000-01-01', tz='UTC')) & ~bulk['key'].isin(excluded)]
bulk['ym'] = bulk['repository'] + '|' + bulk['created'].dt.strftime('%Y-%m')
bulk = bulk.sample(frac=1, random_state=SEED)
bulk['rank'] = bulk.groupby('ym').cumcount()
bulk = bulk[bulk['rank'] < MAX_PER_MONTH].sort_values(['rank', 'ym'])
print(len(bulk), 'issues queued across', bulk['ym'].nunique(), 'repository-months;',
      f"enough budget left for about {int((BUDGET_USD - jev.spent)/(TOK_PER_CALL*PRICE_PER_M_INPUT/1e6)):,} tickets")
bulk[['key', 'repository', 'ym', 'rank']].to_csv(RESULTS/'jev_bulk_queue.csv', index=False)
for start in range(0, len(bulk), 5000):          # chunks so progress is saved and the run can be resumed
    if jev.stop: break
    ch = bulk.iloc[start:start+5000]
    run_jobs(jev, [(r['key'], ticket_state(r), cls_questions()) for _, r in ch.iterrows()], RAW/'bulk_classification.jsonl', workers=16)
print(f'total spent this session: ${jev.spent:.3f} over {jev.calls:,} calls')

## 6. Results — I4 against I1 on the same tickets
Accuracy with Wilson 95% CI, macro-F1, paired McNemar test against I1, risk–coverage, calibration and cost per 1,000 tickets.

In [ ]:
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
ans = {o['job_id']: o for o in read_log(RAW/'rq1_classification.jsonl')}
tok = np.mean([o['usage']['input_tokens'] for o in ans.values()])
rows, preds_out = [], []
for t in QUESTIONS:
    ts = pd.read_csv(TESTS/f'testset_jira_{t}.csv').merge(pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_pred']], on='key')
    ts = ts[ts['key'].isin(ans)]
    a = [ans[k]['answers'][t] for k in ts['key']]
    p = np.array([x['choice'] for x in a]); pmax = np.array([max(x['probabilities'].values()) for x in a]); conf = np.array([x['confidence'] for x in a])
    y = ts['label'].values; ok = p == y; ok1 = ts['i1_pred'].values == y; n = len(y)
    b = int((ok1 & ~ok).sum()); c = int((~ok1 & ok).sum())
    rows.append({'Decision': t, 'Instrument': f'I4 Jev ({MODEL})', 'Test rows': n, 'Accuracy %': round(100*ok.mean(), 1), '95% CI (Wilson)': wilson(int(ok.sum()), n),
                 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3), 'I1 accuracy %': round(100*ok1.mean(), 1),
                 'McNemar p (I4 vs I1)': round(mcnemar([[0, b], [c, 0]], exact=True).pvalue, 4), 'I1 only right': b, 'I4 only right': c,
                 'Coverage % at 98% precision': coverage(pmax, ok, .98), 'Coverage % at 90% precision': coverage(pmax, ok, .90),
                 'ECE (top probability)': ece(pmax, ok), 'Input tokens per ticket (4 decisions)': round(tok),
                 'Cost US$ per 1,000 tickets': round(1000*tok*PRICE_PER_M_INPUT/1e6, 4)})
    preds_out.append(pd.DataFrame({'decision': t, 'key': ts['key'], 'label': y, 'i4_pred': p, 'i4_top_prob': pmax, 'i4_confidence': conf}))
i4 = pd.DataFrame(rows); i4.to_csv(RESULTS/'rq1_jira_i4_jev.csv', index=False)
pd.concat(preds_out).to_csv(TESTS/'i4_predictions_classification.csv', index=False); i4

## 7. Results — re-ranking and consistency

In [ ]:
i1r = pd.read_csv(RESULTS/'rq1_jira_i1_retrieval.csv').set_index('Decision')
ret = []
for t in ('D2_duplicate', 'D6_cause'):
    a = {o['job_id']: o['answers'][t] for o in read_log(RAW/f'rq1_{t}.jsonl')}
    ts = pd.read_csv(TESTS/f'testset_jira_{t}.csv').merge(pd.read_csv(TESTS/f'i1_predictions_{t}.csv'), on='key')
    hit1 = hit1_i1 = n = 0
    for k, answers, top in zip(ts['key'], ts['answers'], ts['i1_top10']):
        if k not in a: continue
        n += 1; truth = set(json.loads(answers)); top = json.loads(top)
        pr = {c: v for c, v in a[k]['probabilities'].items() if c != 'none'}
        best = top[int(max(pr, key=pr.get)[1:]) - 1]
        hit1 += best in truth; hit1_i1 += top[0] in truth
    ret.append({'Decision': t, 'Test rows': n, 'I1 Recall@1 %': round(100*hit1_i1/n, 1), 'I1 + I4 re-rank Recall@1 %': round(100*hit1/n, 1),
                '95% CI': wilson(hit1, n), 'Upper bound (I1 Recall@10) %': float(i1r.loc[t, 'Recall@10 %'])})
ret = pd.DataFrame(ret); ret.to_csv(RESULTS/'rq1_jira_i4_rerank.csv', index=False); display(ret)
agree = []
for rep in (2, 3):
    r2 = {o['job_id'].split('#')[0]: o for o in read_log(RAW/f'rq1_repeat{rep}.jsonl')}
    for t in QUESTIONS:
        same = [ans[k]['answers'][t]['choice'] == r2[k]['answers'][t]['choice'] for k in r2 if k in ans]
        dprob = [abs(max(ans[k]['answers'][t]['probabilities'].values()) - max(r2[k]['answers'][t]['probabilities'].values())) for k in r2 if k in ans]
        agree.append({'Repeat': rep, 'Decision': t, 'Tickets': len(same), 'Same answer %': round(100*np.mean(same), 1), 'Mean |change| in top probability': round(float(np.mean(dprob)), 4)})
agree = pd.DataFrame(agree); agree.to_csv(RESULTS/'rq1_jira_i4_consistency.csv', index=False); agree

## 8. Summary file

In [ ]:
bulk_done = read_log(RAW/'bulk_classification.jsonl') if (RAW/'bulk_classification.jsonl').exists() else []
all_tokens = sum(o['usage']['input_tokens'] for f in RAW.glob('*.jsonl') for o in read_log(f))
q = pd.read_csv(RESULTS/'jev_bulk_queue.csv'); covered = q[q['key'].isin({o['job_id'] for o in bulk_done})]
lines = [f'# Step 08 (I4 Jev) summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', f'Model: {MODEL}',
         f'Total input tokens: {all_tokens:,} = US${all_tokens*PRICE_PER_M_INPUT/1e6:.2f}',
         f'Bulk tickets answered: {len(bulk_done):,} across {covered["ym"].nunique()} repository-months', '',
         i4.to_markdown(index=False), '', ret.to_markdown(index=False), '', agree.to_markdown(index=False)]
(RESULTS/'step08_jev_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))